# C4 SE supervised simulation v1 (`se-supervised-sim-v1`)

S1 catalogue (46 topics). Semester-restricted practice + mock exams.
Additive package `training_se_supervised_v1` — does **not** modify v0.

Targets:
- **A.** Next-attempt correctness
- **B.** Synthetic mock-exam score
- **C.** Synthetic mock-exam pass probability (`PASS_THRESHOLD=0.55` labels only)

Explicitly synthetic. Not real SE / real-exam validation. Do **not** auto-wire into `se-path-sim-v1` without a new contract.

Gates default **False**. Artifacts under `se_supervised_sim_v1` only.


## 1. Dependencies


In [2]:
%pip install -q "numpy>=1.23,<3" "pandas>=1.5,<3" "scikit-learn>=1.2,<2" "scipy>=1.10,<2" "joblib>=1.2,<2" matplotlib tqdm


## 2. Upload & extract `c4-se-supervised-v1-source-colab.zip`


In [3]:
from pathlib import Path
import zipfile, shutil
from google.colab import files

UPLOAD_DIR = Path('/content/c4_se_sup_v1_upload')
EXTRACT_DIR = Path('/content/c4_se_sup_v1_src')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print('Upload c4-se-supervised-v1-source-colab.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded')
zip_name = next(iter(uploaded))
zip_path = UPLOAD_DIR / zip_name
zip_path.write_bytes(uploaded[zip_name])
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)
print('Top:', list(EXTRACT_DIR.iterdir())[:10])


Upload c4-se-supervised-v1-source-colab.zip


Saving c4-se-supervised-v1-source-colab.zip to c4-se-supervised-v1-source-colab.zip
Extracted to /content/c4_se_sup_v1_src
Top: [PosixPath('/content/c4_se_sup_v1_src/ai-services')]


## 3. Drive mount (v1 output root only)


In [4]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts root:', DRIVE_ROOT)
print('Do not write into se_supervised_sim_v0')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifacts root: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1
Do not write into se_supervised_sim_v0


## 4. Imports / versions / paths


In [5]:
import sys, os, json
from pathlib import Path

if 'EXTRACT_DIR' not in globals() or 'DRIVE_ROOT' not in globals():
    raise RuntimeError('Run cells 2 (extract) and 3 (Drive mount) first.')

hits = list(Path(EXTRACT_DIR).rglob('training_se_supervised_v1/protocol.py'))
if not hits:
    raise FileNotFoundError('training_se_supervised_v1/protocol.py not found; re-upload ZIP')
C4_ROOT = hits[0].parents[1].resolve()
sys.path.insert(0, str(C4_ROOT))
os.chdir(C4_ROOT)
print('C4_ROOT', C4_ROOT)

from training_se_supervised_v1.colab_bootstrap import require_symbol
from training_se_supervised_v1.protocol import PROTOCOL_VERSION, GENERATOR_VERSION, DOMAIN_MARKERS
from training_se_supervised_v1.provenance import capture_versions
print('protocol', PROTOCOL_VERSION, GENERATOR_VERSION)
print('markers', DOMAIN_MARKERS)
print(json.dumps(capture_versions(), indent=2))


C4_ROOT /content/c4_se_sup_v1_src/ai-services/c4
protocol se-supervised-sim-v1 se_supervised_gen_v1
markers {'domain': 'se_simulation', 'synthetic': True, 'simulation_only': True, 'real_exam_validation': False, 'protocol': 'se-supervised-sim-v1', 'generator_version': 'se_supervised_gen_v1', 'scope_decision': 'S1'}
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "protocol": "se-supervised-sim-v1",
  "generator_version": "se_supervised_gen_v1",
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "scope_decision": "S1",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "joblib": "1.6.0",
  "scipy": "1.16.3"
}


## 5. Frozen contract + catalogue + thresholds


In [6]:
from training_se_supervised_v1.contract import (
    frozen_contract_dict, PASS_THRESHOLD, CLASSIFICATION_PROB_THRESHOLD, SCENARIOS,
)
from training_se_supervised_v1.inventory import inventory_local_se_exports
from training_se_supervised_v1.catalogue import (
    catalogue_hash, ORDERED_TOPICS, N_TOPICS, Y3S1_TOPICS, Y3S2_TOPICS, validate_catalogue,
)
from training_se_supervised_v1.features_response import response_feature_names
from training_se_supervised_v1.features_exam import exam_feature_names

print(validate_catalogue())
print('n_topics', N_TOPICS, 'y3s1', len(Y3S1_TOPICS), 'y3s2', len(Y3S2_TOPICS))
print('response_feat_dim', len(response_feature_names()))
print('exam_feat_dim', len(exam_feature_names()))
print('PASS_THRESHOLD (synthetic label cutoff only)', PASS_THRESHOLD)
print('CLASSIFICATION_PROB_THRESHOLD (hard-class accuracy only)', CLASSIFICATION_PROB_THRESHOLD)
CONTRACT = frozen_contract_dict(mode='smoke', master_seed=42)
print('scenarios', list(SCENARIOS))
print('catalogue_sha256', catalogue_hash())
print(json.dumps(CONTRACT, indent=2)[:2800])
print(json.dumps(inventory_local_se_exports(C4_ROOT), indent=2))
(DRIVE_ROOT / '_session_contract_preview.json').write_text(
    json.dumps(CONTRACT, indent=2), encoding='utf-8'
)


{'ok': True, 'n_topics': 46, 'catalogue_hash': '95aaba520a28a494e9d6fed0c30de5999b6d47f376368424fb211acb434a1622', 'y3s1': 28, 'y3s2': 18}
n_topics 46 y3s1 28 y3s2 18
response_feat_dim 54
exam_feat_dim 98
PASS_THRESHOLD (synthetic label cutoff only) 0.55
CLASSIFICATION_PROB_THRESHOLD (hard-class accuracy only) 0.5
scenarios ['baseline', 'high_noise', 'easy_bank']
catalogue_sha256 95aaba520a28a494e9d6fed0c30de5999b6d47f376368424fb211acb434a1622
{
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v1",
  "generator_version": "se_supervised_gen_v1",
  "scope_decision": "S1",
  "contract": {
    "protocol": "se-supervised-sim-v1",
    "generator_version": "se_supervised_gen_v1",
    "master_seed": 42,
    "pass_threshold": 0.55,
    "pass_threshold_note": "Pass label = 1[y_exam_score >= 0.55] on the synthetic exam fraction-correct scale. This is a declared synthetic exam-score cutoff only \u2014 not

11222

## 6. Smoke generation + leakage + train/select on validation


In [9]:
from training_se_supervised_v1.pipeline import run_training_and_validation

smoke = run_training_and_validation(
    DRIVE_ROOT,
    mode='smoke',
    master_seed=42,
    run_full=False,
    c4_root=C4_ROOT,
)
SMOKE_RUN_DIR = Path(smoke['run_dir'])
print('smoke run', SMOKE_RUN_DIR)
print('selection', json.dumps(smoke['selection'], indent=2))
print('leakage', smoke['leakage'])


smoke run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1/train_smoke_se-supervised-sim-v1_20261014T001833.548Z
selection {
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v1",
  "generator_version": "se_supervised_gen_v1",
  "scope_decision": "S1",
  "primary_scenario": "baseline",
  "response": {
    "name": "hist_gradient_boosting_classifier",
    "val_metrics": {
      "n": 194,
      "prevalence": 0.4845360824742268,
      "pred_mean": 0.4680896752177372,
      "accuracy": 0.6804123711340206,
      "classification_prob_threshold": 0.5,
      "log_loss": 0.588800480579846,
      "brier": 0.2016805673224379,
      "roc_auc": 0.7570212765957446
    },
    "primary_metric": "log_loss"
  },
  "exam_score": {
    "name": "train_mean",
    "val_metrics": {
      "n": 9,
      "mae": 0.14353426555807505,
      "rmse": 0.17875650410093374,
      "y_mean": 0.4887566137566137,
 

## 7. Validation selection tables


In [10]:
import pandas as pd

require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
sel = json.loads((SMOKE_RUN_DIR / 'selection_val.json').read_text())
resp = json.loads((SMOKE_RUN_DIR / 'comparison_val_response.json').read_text())
exam = json.loads((SMOKE_RUN_DIR / 'comparison_val_exam_score.json').read_text())

df_resp = pd.DataFrame([
    {
        'target': 'response',
        'model': r['name'],
        'val_accuracy': r['val_metrics'].get('accuracy'),
        'val_log_loss': r['val_metrics'].get('log_loss'),
        'val_brier': r['val_metrics'].get('brier'),
        'val_roc_auc': r['val_metrics'].get('roc_auc'),
        'selected': r['name'] == sel['response']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
        'protocol': 'se-supervised-sim-v1',
    }
    for r in resp
])
df_exam = pd.DataFrame([
    {
        'target': 'exam_score',
        'model': r['name'],
        'val_mae': r['val_metrics'].get('mae'),
        'val_rmse': r['val_metrics'].get('rmse'),
        'selected': r['name'] == sel['exam_score']['name'],
        'synthetic': True,
        'domain': 'se_simulation',
        'protocol': 'se-supervised-sim-v1',
    }
    for r in exam
])
display(df_resp)
display(df_exam)
print('0.55 is a labelled synthetic exam-score cutoff, not an institutional rule.')
print('Do not retune generation to force 80–90% accuracy.')


,target,model,val_accuracy,val_log_loss,val_brier,val_roc_auc,selected,synthetic,domain,protocol
0,response,constant_train_prevalence,0.515464,0.692772,0.249812,0.500000,False,True,se_simulation,se-supervised-sim-v1
1,response,logistic_regression,0.628866,0.645243,0.227250,0.670319,False,True,se_simulation,se-supervised-sim-v1
2,response,hist_gradient_boosting_classifier,0.680412,0.588800,0.201681,0.757021,True,True,se_simulation,se-supervised-sim-v1


,target,model,val_mae,val_rmse,selected,synthetic,domain,protocol
0,exam_score,train_mean,0.143534,0.178757,True,True,se_simulation,se-supervised-sim-v1
1,exam_score,ridge_regression,0.193922,0.220265,False,True,se_simulation,se-supervised-sim-v1
2,exam_score,hist_gradient_boosting_regressor,0.144614,0.181871,False,True,se_simulation,se-supervised-sim-v1


0.55 is a labelled synthetic exam-score cutoff, not an institutional rule.
Do not retune generation to force 80–90% accuracy.


## 8. Full training (gated)


In [11]:
RUN_FULL_TRAINING = True

full = run_training_and_validation(
    DRIVE_ROOT,
    mode="full",
    master_seed=42,
    run_full=True,
    c4_root=C4_ROOT,
)

FULL_RUN_DIR = Path(full["run_dir"])

print("Full run:", FULL_RUN_DIR)
print("Selection:")
print(json.dumps(full["selection"], indent=2))
print("Leakage:")
print(json.dumps(full["leakage"], indent=2))

Full run: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1/train_full_se-supervised-sim-v1_20261014T002241.664Z
Selection:
{
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v1",
  "generator_version": "se_supervised_gen_v1",
  "scope_decision": "S1",
  "primary_scenario": "baseline",
  "response": {
    "name": "hist_gradient_boosting_classifier",
    "val_metrics": {
      "n": 13748,
      "prevalence": 0.4820337503636893,
      "pred_mean": 0.4871726873760817,
      "accuracy": 0.6663514693046261,
      "classification_prob_threshold": 0.5,
      "log_loss": 0.6103379736241487,
      "brier": 0.2112751580726346,
      "roc_auc": 0.7252752105613995
    },
    "primary_metric": "log_loss"
  },
  "exam_score": {
    "name": "ridge_regression",
    "val_metrics": {
      "n": 300,
      "mae": 0.06850262447014432,
      "rmse": 0.083333534257756,
      "y_mean": 0.5214329805

## 9. Final-test evaluation (gated; frozen selection only)


In [12]:
from pathlib import Path
import json
from training_se_supervised_v1.pipeline import (
    reload_run,
    run_final_evaluation,
)

FINAL_TARGET_RUN = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_supervised_sim_v1/"
    "train_full_se-supervised-sim-v1_20261014T002241.664Z"
)

assert FINAL_TARGET_RUN.is_dir(), FINAL_TARGET_RUN

reloaded = reload_run(FINAL_TARGET_RUN)
selection = reloaded["selection"]

assert selection["protocol"] == "se-supervised-sim-v1"

print("Run:", FINAL_TARGET_RUN)
print("Frozen validation selections:")
for target in ("response", "exam_score", "pass"):
    print(target, "->", selection[target]["name"])

# Evaluate existing saved models; no retraining.
final = run_final_evaluation(
    FINAL_TARGET_RUN,
    run_final=True,
)

print("\nFinal-test results:")
print(json.dumps(final, indent=2))

Run: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1/train_full_se-supervised-sim-v1_20261014T002241.664Z
Frozen validation selections:
response -> hist_gradient_boosting_classifier
exam_score -> ridge_regression
pass -> logistic_regression

Final-test results:
{
  "domain": "se_simulation",
  "synthetic": true,
  "simulation_only": true,
  "real_exam_validation": false,
  "protocol": "se-supervised-sim-v1",
  "generator_version": "se_supervised_gen_v1",
  "scope_decision": "S1",
  "loaded_selection": {
    "domain": "se_simulation",
    "exam_score": {
      "name": "ridge_regression",
      "primary_metric": "mae",
      "val_metrics": {
        "mae": 0.06850262447014432,
        "n": 300,
        "pred_mean": 0.5060742982815725,
        "pred_std": 0.16148343833291876,
        "rmse": 0.083333534257756,
        "y_mean": 0.5214329805996473,
        "y_std": 0.18242229231459675
      }
    },
    "generator_version": "se_supervised_gen_v1",
    "pass": {
      "n

## 10. Export reminder


In [13]:
from training_se_supervised_v1.export_bundle import verify_manifest

target = globals().get('FULL_RUN_DIR') or globals().get('SMOKE_RUN_DIR')
if target is None:
    print('No run directory in this session')
else:
    export_dir = Path(target) / 'export_bundle'
    print('export_bundle', export_dir)
    print(verify_manifest(export_dir))
    print('Download export_bundle from Drive.')
    print('Marks: domain=se_simulation, synthetic=true, simulation_only=true.')
    print('Do NOT auto-wire into se-path-sim-v1 without an explicit new contract.')


export_bundle /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1/train_full_se-supervised-sim-v1_20261014T002241.664Z/export_bundle
{'ok': True, 'n': 25, 'mismatches': []}
Download export_bundle from Drive.
Marks: domain=se_simulation, synthetic=true, simulation_only=true.
Do NOT auto-wire into se-path-sim-v1 without an explicit new contract.


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
from google.colab import files

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_supervised_sim_v1/"
    "train_full_se-supervised-sim-v1_20261014T002241.664Z"
)

assert (RUN_DIR / "export_bundle").is_dir()

destination = RUN_DIR.parent / "exports"
destination.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
zip_path = shutil.make_archive(
    str(destination / f"c4_se_supervised_v1_bundle_{stamp}"),
    "zip",
    root_dir=RUN_DIR,
    base_dir="export_bundle",
)

print("Package saved:", zip_path)
files.download(zip_path)

Package saved: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_supervised_sim_v1/exports/c4_se_supervised_v1_bundle_20261014T004114837306Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>